# Part 7 — Group-Aware Validation Split

Goal: evaluate the multi-feature regression model on truly unseen users by keeping `user_id` groups separated between training and validation.

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [4]:
from pathlib import Path

ROOT = Path.cwd().parents[1]

print("repo root:", ROOT)

print("\nModel-ready v5 candidates:")
for path in ROOT.rglob("*v5*"):
    if path.is_file():
        print(path.relative_to(ROOT))

print("\nSaved split:")
for path in ROOT.rglob("split_users.csv"):
    if path.is_file():
        print(path.relative_to(ROOT))

repo root: C:\Users\kutay\Desktop\ml-learning

Model-ready v5 candidates:
data\duolingo_flagship_v5.csv

Saved split:
data\split_users.csv


In [5]:
data_path = ROOT / "data" / "duolingo_flagship_v5.csv"
split_path = ROOT / "data" / "split_users.csv"

df = pd.read_csv(data_path)
split_df = pd.read_csv(split_path)

print("df shape:", df.shape)
print("df columns:")
print(df.columns.tolist())

print("\nsplit_df shape:", split_df.shape)
print("split_df columns:")
print(split_df.columns.tolist())

display(split_df.head())

df shape: (16382, 18)
df columns:
['practice_time', 'user_id', 'ui_language', 'learning_language', 'surface_form', 'lemma', 'pos', 'grammar_tags', 'lag_days', 'history_seen', 'history_correct', 'session_seen', 'session_correct', 'p_recall', 'lexeme_id', 'lag_days_log', 'history_accuracy', 'difficulty_rank_in_language']

split_df shape: (2500, 2)
split_df columns:
['user_id', 'split']


,user_id,split
0,u:0Bk,test
1,u:27Z,test
2,u:F-1,test
3,u:FHQ,test
4,u:QMy,test


In [6]:
print(split_df["split"].value_counts())

split
cv      2125
test     375
Name: count, dtype: int64


In [9]:
cv_users = set(split_df.loc[split_df["split"] == "cv", "user_id"])

df_cv = df[df["user_id"].isin(cv_users)].copy()

print("CV pool shape:", df_cv.shape)
print("CV pool unique users:", df_cv["user_id"].nunique())

print("Contains test users:", df_cv["user_id"].isin(split_df.loc[split_df["split"] == "test", "user_id"]).any())

CV pool shape: (14438, 18)
CV pool unique users: 2125
Contains test users: False


In [10]:
features = [
    "lag_days",
    "history_seen",
    "history_correct",
    "history_accuracy",
    "lag_days_log"
]

X = df_cv[features]
y = df_cv["p_recall"]
groups = df_cv["user_id"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("groups:", groups.nunique())

X shape: (14438, 5)
y shape: (14438,)
groups: 2125


In [11]:
gkf = GroupKFold(n_splits=5)

train_idx, val_idx = next(gkf.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

groups_train = groups.iloc[train_idx]
groups_val = groups.iloc[val_idx]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

print("train users:", groups_train.nunique())
print("val users:", groups_val.nunique())

X_train: (11550, 5)
X_val: (2888, 5)
train users: 1700
val users: 425


In [12]:
train_users = set(groups_train)
val_users = set(groups_val)

overlap = train_users & val_users

print("overlap count:", len(overlap))
print("overlap users:", overlap)

assert len(overlap) == 0

overlap count: 0
overlap users: set()


In [13]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("Validation RMSE:", rmse)

Validation RMSE: 0.28485344103032856


In [15]:
mean_baseline_pred = np.full(len(y_val), y_train.mean())
baseline_rmse = np.sqrt(mean_squared_error(y_val, mean_baseline_pred))

print("Mean baseline RMSE:", baseline_rmse)
print("Linear model RMSE:", rmse)

Mean baseline RMSE: 0.2874626642713031
Linear model RMSE: 0.28485344103032856


## Conclusion

- A random row-level split would allow the same `user_id` to appear in both training and validation sets, causing group leakage.
- `GroupKFold` keeps all rows from the same user in the same fold.
- For the selected split, train and validation users were completely disjoint:

$$
\text{train users} \cap \text{validation users} = \emptyset
$$

- Preprocessing was placed inside a `Pipeline`, so `StandardScaler` was fit only on the training fold.
- Mean baseline RMSE: `0.28746`
- Linear model validation RMSE: `0.28485`

The linear model slightly outperformed the mean baseline on unseen users.

This score is more trustworthy than an in-sample score because the model was evaluated on users it had never seen during training.

However, this is only one GroupKFold split. A single split may be unusually easy or difficult, so the next step is 5-fold group-aware cross-validation.